In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

In [3]:
!dir

 Volume in drive C is OS
 Volume Serial Number is 1E87-BAE1

 Directory of C:\Users\casus\OneDrive\Desktop\AERO ORGANISED\MY WORK, PROJECTS ETC\RESEARCH & PATENTS\ANUP SIR\Optimising_conference_data

07/10/2025  15:19    <DIR>          .
07/10/2025  14:33    <DIR>          ..
07/10/2025  14:36    <DIR>          .ipynb_checkpoints
07/10/2025  14:33            16,685 CL & CD Graphs of Injection & Suction.xlsx
07/10/2025  15:00               941 conference_injection_only.csv
07/10/2025  15:16               941 conference_suction_only.csv
07/10/2025  15:06            14,539 injection_only.csv
07/10/2025  14:38               617 Optimising_conference_work.ipynb
06/10/2025  11:09           457,665 Performance_improvement_of_horizontal_axis_wind_turbine_using_active_control_methods.pdf
07/10/2025  15:19            14,972 suction_only.csv
               7 File(s)        506,360 bytes
               3 Dir(s)  190,766,604,288 bytes free


In [4]:
injection_df = pd.read_csv('./injection_only.csv')
injection_df.columns = ['AOA', 'injection_location', 'injection_slot_size', 'cl', 'cd', 'Re', 'chord_length']
injection_df

,AOA,injection_location,injection_slot_size,cl,cd,Re,chord_length
0,-20.0,4,0.5,-0.65,0.280,1000000,460
1,-19.5,4,0.5,-0.66,0.265,1000000,460
2,-19.0,4,0.5,-0.67,0.250,1000000,460
3,-18.5,4,0.5,-0.68,0.235,1000000,460
4,-18.0,4,0.5,-0.69,0.220,1000000,460
...,...,...,...,...,...,...,...
679,25.5,50,2.0,1.77,1.052,2000000,600
680,26.0,50,2.0,1.74,1.084,2000000,600
681,26.5,50,2.0,1.71,1.116,2000000,600
682,27.0,50,2.0,1.68,1.148,2000000,600


In [5]:
# setting up the data
data_df = injection_df.drop(columns=['cl', 'cd'])
data_df
injection_data = np.array(data_df)
injection_data

array([[-2.00e+01,  4.00e+00,  5.00e-01,  1.00e+06,  4.60e+02],
       [-1.95e+01,  4.00e+00,  5.00e-01,  1.00e+06,  4.60e+02],
       [-1.90e+01,  4.00e+00,  5.00e-01,  1.00e+06,  4.60e+02],
       ...,
       [ 2.65e+01,  5.00e+01,  2.00e+00,  2.00e+06,  6.00e+02],
       [ 2.70e+01,  5.00e+01,  2.00e+00,  2.00e+06,  6.00e+02],
       [ 2.75e+01,  5.00e+01,  2.00e+00,  2.00e+06,  6.00e+02]],
      shape=(684, 5))

In [6]:
#setting up the targets
injection_cl = np.array(injection_df['cl'])
injection_cd = np.array(injection_df['cd'])
# injection_cl

In [9]:
# splitting the data for cl and and cd separately
from sklearn.model_selection import train_test_split

X_train_cl, X_test_cl, y_train_cl, y_test_cl = train_test_split(injection_data, injection_cl, random_state=0)
X_train_cd, X_test_cd, y_train_cd, y_test_cd = train_test_split(injection_data, injection_cd, random_state=0)

# X_train_cd
# X_train_cl

### Data Scaling

In [10]:
# scaling the data
from sklearn.preprocessing import StandardScaler

scaler_cl = StandardScaler().fit(X_train_cl)
scaler_cd = StandardScaler().fit(X_train_cd)
# scaling the cl data

X_train_cl_scaled = scaler_cl.transform(X_train_cl)
X_test_cl_scaled = scaler_cl.transform(X_test_cd)

# scaling the cd data
X_train_cd_scaled = scaler_cd.transform(X_train_cd)
X_test_cd_scaled = scaler_cd.transform(X_test_cd)

# X_train_cd_scaled
# X_train_cl_scaled


### Training the models

In [17]:
from sklearn.neural_network import MLPRegressor

mlp_cl = MLPRegressor(random_state=0, early_stopping=True, max_iter=300).fit(X_train_cl_scaled, y_train_cl)

mlp_cd = MLPRegressor(random_state=0, early_stopping=True, hidden_layer_sizes=(100, 75), max_iter=250).fit(X_train_cd_scaled, y_train_cd)

print("CL prediction accuracy")
print(f"Cl training score: {mlp_cl.score(X_train_cl_scaled, y_train_cl)}")
print(f"Cl Test score: {mlp_cl.score(X_test_cl_scaled, y_test_cl)}")


print("\nCD prediction accuracy")
print(f"Cd training score: {mlp_cd.score(X_train_cd_scaled, y_train_cd)}")
print(f"Cd Test score: {mlp_cd.score(X_test_cd_scaled, y_test_cd)}")



CL prediction accuracy
Cl training score: 0.9892981625284556
Cl Test score: 0.9901354385187725

CD prediction accuracy
Cd training score: 0.9974825363214909
Cd Test score: 0.9963611957407597


###  Optimising CL

In [22]:
angles = np.arange(0, 28, 0.5)
angles = np.arange(0, 5, 1)
# angles
optimised_cl_values = []
optimised_injection_locations= []
optimised_injection_slot_sizes= []
optimised_Re_values= []
optimised_chord_length_values= []

In [23]:
import skopt
from skopt.optimizer import gp_minimize
from skopt.space import Real, Integer, Categorical

for angle in angles:
    print(f'optimising for {angle} deg', end=',')
    def objective_function_cl(params):
        aoa = angle
        injection_location = 50
        injection_slot_size, Re, chord_length  = params
        X_input = [[aoa, injection_location, injection_slot_size, Re, chord_length]]
        X_input_scaled = scaler_cl.transform(X_input)
        predicted_cl = mlp_cl.predict(X_input_scaled)


        return -predicted_cl[0]

    space = [
        # Integer(2, 50, name='injection_location'),
        Real(0.5, 2, name='injection_slot_size'),
        Integer(1000000, 2000000, name='Re'),
        Integer(460, 600, name='chord_length')    
    ]

    result = gp_minimize(
        func=objective_function_cl,
        dimensions=space,
        n_calls = 30,
        random_state = 0
    )
    optimised_cl_values.append(-result.fun)
    # optimised_injection_locations.append(result.x[0])
    optimised_injection_slot_sizes.append(result.x[0])
    optimised_Re_values.append(result.x[1])
    optimised_chord_length_values.append(result.x[2])

    print(f' {angle} done!')
    

optimising for 0 deg, 0 done!
optimising for 1 deg, 1 done!
optimising for 2 deg, 2 done!
optimising for 3 deg, 3 done!
optimising for 4 deg, 4 done!


In [26]:
output_dict = {
    'aoa' : angles,
    # 'optimal_injection_loc' : optimised_injection_locations,
    'optimal_injection_slot_size' : optimised_injection_slot_sizes,
    're_values' : optimised_Re_values,
    'chord_length' : optimised_chord_length_values
}
output_dict

# -result.fun

{'aoa': array([0, 1, 2, 3, 4]),
 'optimal_injection_slot_size': [2.0, 2.0, 2.0, 2.0, 2.0],
 're_values': [np.int64(1000000),
  np.int64(1000000),
  np.int64(1000000),
  np.int64(1000000),
  np.int64(1000000)],
 'chord_length': [np.int64(600),
  np.int64(600),
  np.int64(600),
  np.int64(600),
  np.int64(600)]}